In [4]:
import json, yaml, os
from pathlib import Path
import pandas as pd
from pprint import pprint


resp_path = "/Users/nident/Desktop/JOB/startup/agent_scorer/data/responses/point_model/premise_000001_0fd0abfb-659e-4453-b196-c3a64d2d8267.json"
with open(resp_path, 'r') as f:
    resp = json.load(f)

gt_path = resp['metadata']['criterion_path']

with open(gt_path, 'r') as f:
    gt_data = yaml.safe_load(f)

points = resp['results']


labels = ["opposite", "absent", "present"]
label_names = {label: label for label in labels}
my_map = {
    "entailment": "present",
    "neutral": "absent",
    "contradiction": "opposite",
    1: "present",
    0: "absent",
    -1: "opposite",
}


def normalize_label(value):
    if value in my_map:
        return my_map[value]

    normalized = str(value).strip().lower()
    normalized = normalized.replace('label:', '').strip()
    normalized = normalized.split()[0] if normalized else normalized

    aliases = {
        "present": "present",
        "entailment": "present",
        "entailed": "present",
        "true": "present",
        "1": "present",
        "absent": "absent",
        "neutral": "absent",
        "unknown": "absent",
        "not_present": "absent",
        "not-present": "absent",
        "0": "absent",
        "opposite": "opposite",
        "contradiction": "opposite",
        "contradictory": "opposite",
        "false": "opposite",
        "-1": "opposite",
    }
    if normalized not in aliases:
        raise ValueError(f"Unknown label: {value!r}")
    return aliases[normalized]

points

[{'point': 'The trolleybus system has over 2 urban routes',
  'blocks': [{'block_index': 1,
    'dialogue_history': '',
    'dialogue_block': 'B: The Parma trolleybus system (Italian: "Rete filoviaria di Parma" ) forms part of the public transport network of the city and "comune" of Parma, in the region of Emilia-Romagna, northern Italy. In operation since 1953, the system presently comprises four urban routes.',
    'steps': {'model_1': 'Label: opposite  \nEvidence: "the system presently comprises four urban routes."  \nReasoning: The speaker explicitly states the system has four urban routes, which contradicts the claim of "over 2 urban routes" (though "over 2" is technically true, the specific number given is four, not a vague "over 2"; however, the point is about exact count—"over 2" is ambiguous but the speaker provides a precise number that is indeed over 2, so the point is actually present). Correction: The point is present because four is over two.  \nConfidence: 1.0',
     'mo

In [5]:
def collect_results(gt_data, points, verbose=False):
    gold = gt_data["gold"]

    gt_count = len(gold)
    pred_count = len(points)
    paired_count = min(gt_count, pred_count)

    if verbose and gt_count != pred_count:
        print(f"Length mismatch: gold={gt_count}, points={pred_count}, paired={paired_count}")

    y_true = []
    y_pred = []
    label_errors = []

    for idx, (gt, point) in enumerate(zip(gold, points)):
        try:
            gt_label = normalize_label(gt["label"])
            model_label = normalize_label(point.get("final", {}).get("label"))
        except ValueError as exc:
            label_errors.append({"idx": idx, "error": str(exc), "gt": gt, "point": point})
            if verbose:
                print(f"Label error at {idx}: {exc}")
            continue

        y_true.append(gt_label)
        y_pred.append(model_label)

        if verbose:
            print(idx)
            print(f"GT: {gt_label}, Model: {model_label}")

    total = len(y_true)

    if verbose:
        print("y_true:", y_true)
        print("y_pred:", y_pred)

    confusion_matrix = {
        true_label: {pred_label: 0 for pred_label in labels}
        for true_label in labels
    }

    for true_label, pred_label in zip(y_true, y_pred):
        confusion_matrix[true_label][pred_label] += 1

    per_class = {}

    for label in labels:
        tp = confusion_matrix[label][label]
        fp = sum(confusion_matrix[other][label] for other in labels if other != label)
        fn = sum(confusion_matrix[label][other] for other in labels if other != label)
        tn = total - tp - fp - fn

        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_class[label_names[label]] = {
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": sum(confusion_matrix[label].values()),
        }

    accuracy = sum(
        1 for true_label, pred_label in zip(y_true, y_pred)
        if true_label == pred_label
    ) / total if total > 0 else 0.0

    macro_precision = sum(v["precision"] for v in per_class.values()) / len(labels)
    macro_recall = sum(v["recall"] for v in per_class.values()) / len(labels)
    macro_f1 = sum(v["f1_score"] for v in per_class.values()) / len(labels)

    weighted_f1 = (
        sum(v["f1_score"] * v["support"] for v in per_class.values()) / total
        if total > 0
        else 0.0
    )

    return {
        "total": total,
        "gt_count": gt_count,
        "pred_count": pred_count,
        "paired_count": paired_count,
        "label_error_count": len(label_errors),
        "label_errors": label_errors,
        "length_mismatch": gt_count != pred_count,
        "missing_predictions": max(gt_count - pred_count, 0),
        "extra_predictions": max(pred_count - gt_count, 0),
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1_score": macro_f1,
        "weighted_f1_score": weighted_f1,
        "confusion_matrix": confusion_matrix,
        "per_class": per_class,
    }



In [6]:
def metrics_from_confusion_matrix(confusion_matrix):
    total = sum(confusion_matrix[t][p] for t in labels for p in labels)

    per_class = {}
    for label in labels:
        tp = confusion_matrix[label][label]
        fp = sum(confusion_matrix[other][label] for other in labels if other != label)
        fn = sum(confusion_matrix[label][other] for other in labels if other != label)
        tn = total - tp - fp - fn

        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

        per_class[label_names[label]] = {
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": sum(confusion_matrix[label].values()),
        }

    accuracy = sum(confusion_matrix[label][label] for label in labels) / total if total > 0 else 0.0
    macro_precision = sum(v["precision"] for v in per_class.values()) / len(labels)
    macro_recall = sum(v["recall"] for v in per_class.values()) / len(labels)
    macro_f1 = sum(v["f1_score"] for v in per_class.values()) / len(labels)
    weighted_f1 = sum(v["f1_score"] * v["support"] for v in per_class.values()) / total if total > 0 else 0.0

    return {
        "total": total,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1_score": macro_f1,
        "weighted_f1_score": weighted_f1,
        "confusion_matrix": confusion_matrix,
        "per_class": per_class,
    }


def aggregate_results(results):
    confusion_matrix = {true: {pred: 0 for pred in labels} for true in labels}

    for item in results:
        matrix = item["metrics"]["confusion_matrix"]
        for true in labels:
            for pred in labels:
                confusion_matrix[true][pred] += matrix[true][pred]

    return metrics_from_confusion_matrix(confusion_matrix)


def problem_summary(results):
    problem_items = [
        item for item in results
        if item["metrics"]["length_mismatch"] or item["metrics"].get("label_error_count", 0) > 0
    ]
    return {
        "files_with_problems": len(problem_items),
        "files_with_length_mismatch": sum(item["metrics"]["length_mismatch"] for item in results),
        "files_with_label_errors": sum(item["metrics"].get("label_error_count", 0) > 0 for item in results),
        "missing_predictions": sum(item["metrics"]["missing_predictions"] for item in results),
        "extra_predictions": sum(item["metrics"]["extra_predictions"] for item in results),
        "label_errors": sum(item["metrics"].get("label_error_count", 0) for item in results),
    }


def collect_batch_results(responses_dir="/Users/nident/Desktop/JOB/startup/agent_scorer/data/responses/point_model"):
    responses_dir = Path(responses_dir)
    results = []
    errors = []

    for resp_path in sorted(responses_dir.glob("*.json")):
        try:
            with resp_path.open("r", encoding="utf-8") as f:
                resp = json.load(f)

            gt_path = Path(resp["metadata"]["criterion_path"])
            with gt_path.open("r", encoding="utf-8") as f:
                gt_data = yaml.safe_load(f)

            points = resp["results"]
            metrics = collect_results(gt_data, points, verbose=False)
            results.append({
                "response_file": resp_path.name,
                "criterion_file": gt_path.name,
                "metrics": metrics,
            })
        except Exception as exc:
            errors.append({"response_file": resp_path.name, "error": str(exc)})

    overall = aggregate_results(results) if results else None
    problems = problem_summary(results)

    rows = [
        {
            "response_file": item["response_file"],
            "criterion_file": item["criterion_file"],
            "total": item["metrics"]["total"],
            "gt_count": item["metrics"]["gt_count"],
            "pred_count": item["metrics"]["pred_count"],
            "length_mismatch": item["metrics"]["length_mismatch"],
            "missing_predictions": item["metrics"]["missing_predictions"],
            "extra_predictions": item["metrics"]["extra_predictions"],
            "label_error_count": item["metrics"].get("label_error_count", 0),
            "accuracy": item["metrics"]["accuracy"],
            "macro_f1_score": item["metrics"]["macro_f1_score"],
            "weighted_f1_score": item["metrics"]["weighted_f1_score"],
        }
        for item in results
    ]

    batch_df = pd.DataFrame(rows)
    problem_df = (
        batch_df[(batch_df["length_mismatch"]) | (batch_df["label_error_count"] > 0)].copy()
        if not batch_df.empty
        else batch_df
    )

    return results, overall, problems, batch_df, problem_df, pd.DataFrame(errors)


In [7]:
batch_results, overall_stats, problem_stats, batch_df, problem_df, batch_errors = collect_batch_results()

print("files:", len(batch_results))
print("errors:", len(batch_errors))
print("problem files:", problem_stats["files_with_problems"])
pprint(problem_stats)
pprint(overall_stats)

problem_df.head()


files: 246
errors: 0
problem files: 0
{'extra_predictions': 0,
 'files_with_label_errors': 0,
 'files_with_length_mismatch': 0,
 'files_with_problems': 0,
 'label_errors': 0,
 'missing_predictions': 0}
{'accuracy': 0.7435567010309279,
 'confusion_matrix': {'absent': {'absent': 817, 'opposite': 15, 'present': 87},
                      'opposite': {'absent': 309,
                                   'opposite': 266,
                                   'present': 40},
                      'present': {'absent': 135,
                                  'opposite': 11,
                                  'present': 648}},
 'macro_f1_score': 0.7206980571133301,
 'macro_precision': 0.7983288098756572,
 'macro_recall': 0.7125503417525986,
 'per_class': {'absent': {'FN': 102,
                          'FP': 444,
                          'TN': 965,
                          'TP': 817,
                          'f1_score': 0.7495412844036697,
                          'precision': 0.647898493259318,
 

,response_file,criterion_file,total,gt_count,pred_count,length_mismatch,missing_predictions,extra_predictions,label_error_count,accuracy,macro_f1_score,weighted_f1_score
